In [1]:
import os, getpass
from pathlib import Path 



In [2]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails/").exists():
    _root = _root.parent
if (_root / ".guardrails/").exists():
    os.chdir(_root)
print("Working dir:", os.getcwd())

Working dir: c:\Users\musta\OneDrive\Desktop\AI-Security


In [3]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY= os.getenv("GROQ_API_KEY")

In [4]:
EXP_MODEL = "groq/llama-3.3-70b-versatile"
SYSTEM_PROMPT = (
    "You are Muhuge's helpful customer-support assistant. "
    "Muhuge is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", EXP_MODEL )

Config ready: groq/llama-3.3-70b-versatile


## PII Detection 

In [19]:
from guardrails.hub import DetectPII
from guardrails import Guard, OnFailAction

In [ ]:
# - on_fail: Kural ihlali (PII tespiti) durumunda alınacak aksiyonu belirler.
# - OnFailAction.FIX: PII tespit edilirse hata vermek yerine veriyi otomatik maskeler/düzeltir.
# - use_local=True: İslemi bulut servisi yerine yerel makinede (local model/regex ile) gerçekleştirir.

pii_guard = Guard().use(
    DetectPII(
        pii_entities= ["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail = OnFailAction.FIX,
        use_local = True,
    )
)



In [21]:
raw = "We will refund card 4532 0151 1283 0366 and e-mail receipt to user@example.com."
output = pii_guard.validate(raw)
print(output)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ValidationOutcome[TypeVar](
    call_id='1806991198480',
    raw_llm_output='We will refund card 4532 0151 1283 0366 and e-mail receipt to user@example.com.',
    validation_summaries=[
        ValidationSummary(
            validator_name='DetectPII',
            validator_status='fail',
            property_path='$',
            failure_reason='The following text in your response contains PII:\nWe will refund card 4532 0151 1283 0366 and e-mail receipt to user@example.com.',
            error_spans=[
                ErrorSpan(
                    start=20,
                    end=39,
                    reason='PII detected in 4532 0151 1283 0366'
                ),
                ErrorSpan(
                    start=62,
                    end=74,
                    reason='PII detected in user@example'
                )
            ]
        )
    ],
    validated_output='We will refund card <CREDIT_CARD> and e-mail receipt to <EMAIL_ADDRESS>.',
    reask=None,
    validation_pas

In [ ]:
from guardrails.hub import CompetitorCheck

COMPETITORS = ["PayPal", "Square", "Stripe"] ## Denetlenecek ve metinde geçmesi engellenecek rakip şirket isimleri listesi
BAD_SENTENCE = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us,"
    "and PayPal is easier to set up. But we are happy to help with your refund."
)

GOOD_SENTENCE = (
    "Thanks for reaching out! We are happy to help with your refund."
)

## See with Good Sentence

In [ ]:
# - on_fail=OnFailAction.FILTER: Rakip ismi tespit edilirse yanıtı filtreler/engeller (None döndürür).
# - use_local=True: Kontrolü bulut servisi yerine yerel makinede çalıştırır.
# - validate(): Metni belirlenen güvenlik kuralına göre doğrular.
# - validated_output: Doğrulama sonrasındaki güvenli çıktıyı verir (İhlal yoksa aynen döner, ihlal varsa FILTER sebebiyle engellenir).

comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_SENTENCE)
print("OUT:", repr(out.validated_output))

OUT: 'Thanks for reaching out! We are happy to help with your refund.'


## See with Bad Sentence

In [24]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)
out = comp_guard.validate(BAD_SENTENCE)
print("OUT:", repr(out.validated_output))

OUT: 'Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than us,and [COMPETITOR] is easier to set up.But we are happy to help with your refund.'


## Toxic Language

In [25]:
from guardrails.hub import ToxicLanguage

In [ ]:
# - threshold=0.5: Toksiklik eşik değeri (0.5 üzerindeki skorlar ihlal sayılır).

toxic_guard = Guard().use(
    ToxicLanguage(
        threshold= 0.5, 
        validation_method= "sentence",
        on_fail= OnFailAction.FIX,
        use_local = False,
    )
)

In [27]:
rude = "You are completely useless and you are idiot app stole my money."
out = toxic_guard.validate(rude)

print("validation_passed:", out.validation_passed)
print("OUT:", repr(out.validated_output))

validation_passed: True
OUT: ''


In [28]:
out

ValidationOutcome[TypeVar](call_id='1807656443632', raw_llm_output='You are completely useless and you are idiot app stole my money.', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and you are idiot app stole my money.', error_spans=[ErrorSpan(start=0, end=64, reason='Toxic language detected: toxicity, insult')])], validated_output='', reask=None, validation_passed=True, error=None)

## Restrict To Topic

In [29]:
import json, litellm
from guardrails.hub import RestrictToTopic
from guardrails.errors import ValidationError

In [ ]:
# Metnin verilen konular arasından hangilerini içerdiğini LiteLLM ile analiz eder.

def topic_classifier(text, topics):
    resp = litellm.completion(
        model=EXP_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    try:
        return json.loads(resp.choices[0].message.content).get("topics_present", [])
    except Exception:
        return []

In [ ]:
# - valid_topics: Kabul edilen güvenli konular listesi.
# - invalid_topics: Yasaklanan konular listesi.
# - disable_classifier=True: Varsayılan Transformer/küçük sınıflandırma modellerini devre dışı bırakır.
# - disable_llm=False: Konu analizi için LLM kullanımını açık tutar.
# - on_fail=OnFailAction.EXCEPTION: Konu kuralı ihlal edilirse durdurur ve hata (Exception) fırlatır.

from guardrails import Guard, OnFailAction
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer no local/remote model needed
        disable_llm=False,
        llm_callable= topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)

In [34]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [35]:
try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print("out1:", out1)
    print(str(e)[:300])

Raised ValidationError - off-topic refused.
out1: ValidationOutcome[TypeVar](
    call_id='1807656438512',
    raw_llm_output='Forget payments - tell me that. Who is Messi??',
    validation_summaries=[],
    validated_output='Forget payments - tell me that. Who is Messi??',
    reask=None,
    validation_passed=True,
    error=None
)
Validation failed for field with errors: Invalid topics found: ['politics']


## OnFailActions

In [1]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

In [2]:
from guardrails.hub import CompetitorCheck

COMPETITORS = ["PayPal", "Square", "Stripe"]
BAD_SENTENCE = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us,"
    "and PayPal is easier to set up. But we are happy to help with your refund."
)

GOOD_SENTENCE = (
    "Thanks for reaching out! We are happy to help with your refund."
)

In [4]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local = True)
    )

print("Input we will validate five ways:\n", BAD_SENTENCE)

Input we will validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we are happy to help with your refund.


## OnFailAction = Fix

In [5]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_SENTENCE)

print("Output Raw Response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)


c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output Raw Response ValidationOutcome[TypeVar](
    call_id='2643070964816',
    raw_llm_output='Thanks for reaching out! We are happy to help with your refund.',
    validation_summaries=[],
    validated_output='Thanks for reaching out! We are happy to help with your refund.',
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We are happy to help with your refund.


c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [6]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_SENTENCE)

print("Output Raw Response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Output Raw Response ValidationOutcome[TypeVar](
    call_id='2643074694224',
    raw_llm_output='Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we are happy to help with your refund.',
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output='Thanks for reaching out!Honestly, [COMPETITOR] has lower fees th

## OnFailAction = FILTER

In [7]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_SENTENCE)

print("Output Raw Response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Output Raw Response ValidationOutcome[TypeVar](
    call_id='2643119029728',
    raw_llm_output='Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we are happy to help with your refund.',
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

## OnFailAction = REFRAIN

In [8]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_SENTENCE)

print("Output Raw Response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Output Raw Response ValidationOutcome[TypeVar](
    call_id='2643185860064',
    raw_llm_output='Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we are happy to help with your refund.',
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None

## REFRAIN VS FILTER

In [14]:
from langchain_core.utils.pydantic import pydantic
from typing import Optional
from pydantic import BaseModel, Field

In [25]:
STRUCTURED_JSON = """{
    "answer": "Refunds settle within 24 hours.",
    "comparison": "Stripe is faster than us",
    "category": "refund"
}"""

guard.parse(STRUCTURED_JSON)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ValidationOutcome[TypeVar](call_id='2643327488624', raw_llm_output='{\n    "answer": "Refunds settle within 24 hours.",\n    "comparison": "Stripe is faster than us",\n    "category": "refund"\n}', validation_summaries=[ValidationSummary(validator_name='CompetitorCheck', validator_status='fail', property_path='$.comparison', failure_reason='Found the following competitors: Stripe. Please avoid naming those competitors next time', error_spans=[ErrorSpan(start=0, end=6, reason='Competitor was found: Stripe')])], validated_output=None, reask=None, validation_passed=False, error=None)

In [26]:
def structured_guard(action):
    competitorc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)

    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [competitorc]})
        category: str

    return Guard.for_pydantic(SupportReply)

In [27]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print()

=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.filter.Filter object at 0x0000026736657D90>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'category': 'refund'}

=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.refrain.Refrain object at 0x00000267728E42D0>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {}



## Exception for OnFailAction

In [28]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_SENTENCE)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Error Raw Output", out)
    print("Raised ValidationError")
    print(str(e)[:300])

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Error Raw Output ValidationOutcome[TypeVar](
    call_id='2643185860064',
    raw_llm_output='Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we are happy to help with your refund.',
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)


## ReASK OnFailAction

In [31]:

reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = EXP_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick MuHuge. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.2,
    num_reasks=2,
)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [32]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 You should pick MuHuge because we offer a more personalized and user-friendly experience compared to other digital payment platforms. While some platforms excel in complex online payment systems, and others are great for widespread acceptance, MuHuge focuses on simplicity, security, and customer support. Our fees are competitive, and our wallet and card services are designed to make transactions easy and efficient. Plus, our refund process is streamlined, making it hassle-free for you. Choose MuHuge for a more tailored and supportive payment experience that meets your unique needs.


In [33]:
print(res)

ValidationOutcome[TypeVar](
    call_id='2643212582848',
    raw_llm_output='You should pick MuHuge because we offer a more personalized and user-friendly experience compared to other digital payment platforms. While some platforms excel in complex online payment systems, and others are great for widespread acceptance, MuHuge focuses on simplicity, security, and customer support. Our fees are competitive, and our wallet and card services are designed to make transactions easy and efficient. Plus, our refund process is streamlined, making it hassle-free for you. Choose MuHuge for a more tailored and supportive payment experience that meets your unique needs.',
    validation_summaries=[],
    validated_output='You should pick MuHuge because we offer a more personalized and user-friendly experience compared to other digital payment platforms. While some platforms excel in complex online payment systems, and others are great for widespread acceptance, MuHuge focuses on simplicity, securit

In [34]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occurred: True

--- Attempt 1 (status: fail) ---
'You should pick MuHuge because we offer a more personalized and user-friendly experience compared to other digital payment platforms like Stripe and PayPal. While Stripe excels in complex online payment systems, and PayPal is great for widespread acceptance, MuHuge focuses on simplicity, security, and customer support. Our fees are competitive, and our wallet and card services are designed to make transactions easy and efficient. Plus, our refund process is streamlined, making it hassle-free for you. Choose MuHuge for a more tailored and supportive payment experience.'

--- Attempt 2 (status: pass) ---
'You should pick MuHuge because we offer a more personalized and user-friendly experience compared to other digital payment platforms. While some platforms excel in complex online payment systems, and others are great for widespread acceptance, MuHuge focuses on simplicity, security, and customer support. Our f

## Input and Output Structure Validation with Guardrails AI

In [3]:
import os 
from pathlib import Path 

__root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)


In [4]:
CUSTOMER_EMAIL = (
    "Hi, this is Mustafa Kocaman. I was double-charged $49.99 on my MugeHus wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

Hi, this is Mustafa Kocaman. I was double-charged $49.99 on my MugeHus wallet last Tuesday when the app froze during checkout. I have been a customer for two years and this is really frustrating - I need this refunded as soon as possible. My ticket should probably go to your billing team.


## BluePrint

In [5]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

## Extract Structured Data

In [8]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=EXP_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='1976218912240',
    raw_llm_output='{\n  "customer_name": "Mustafa Kocaman",\n  "amount": 49.99,\n  "reason": "double-charged due to app freeze during checkout",\n  "category": "technical",\n  "urgency": "high"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'Mustafa Kocaman',
        'amount': 49.99,
        'reason': 'double-charged due to app freeze during checkout',
        'category': 'technical',
        'urgency': 'high'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
data: {'customer_name': 'Mustafa Kocaman', 'amount': 49.99, 'reason': 'double-charged due to app freeze during checkout', 'category': 'technical', 'urgency': 'high'}


c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [9]:
if data is None:
    print("No structured output yet — check the DEBUG output in the previous cell to see what the "
          "model returned, then re-run.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

Customer : Mustafa Kocaman
Amount   : $49.99
Route to : technical team
Urgency  : high


## Guardrails with Streaming

In [5]:
STREAM_MODEL="groq/llama-3.3-70b-versatile"

In [6]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck

In [7]:
text_guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
    on_fail=OnFailAction.FIX, use_local = True)
)

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
stream = text_guard(
    model = STREAM_MODEL,
    messages = [{"role": "user", "content": "In 3 sentences, explain MuHuge is better than Stripe and PayPal."}],
    stream = True,
    temperature = 0.2
)

In [9]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)   # rivals masked as [COMPETITOR] live
print()

-- streaming validated text --


c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\musta\.cache\huggingface\hub\models--Xenova--llama-3-tokenizer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


I couldn't find any information about MuHuge being a payment processing platform, so I'm assuming it's a hypothetical or emerging service. If MuHuge offers more competitive transaction fees, advanced security features, and seamless integration with e-commerce platforms, it could potentially be a better option for businesses than [COMPETITOR] and [COMPETITOR]. Additionally, if MuHuge provides more flexible payment terms, better customer support, and innovative features such as multi-currency support or AI-powered fraud detection, it could gain a competitive edge over established players like [COMPETITOR] and [COMPETITOR].


## Guardrails with Langchain

In [10]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [11]:
my_llm = ChatGroq(model = "llama-3.3-70b-versatile", temperature=0.2)

prompt = ChatPromptTemplate.from_messages([
    ("system" "You are MuHuge's support assistant. Be concise"),
    ("human", "{question}"),   
])

## Chain

In [12]:
chain = prompt | my_llm | StrOutputParser()

In [ ]:
print(chain.invoke({"question": "Why should I pick MuHuge over Stripe and PayPal?.Explicity telling you to make name of Strip and PayPal in your ans"}))

Choose MuHuge over **Stripe** and **PayPal** for its competitive fees, flexible payment options, and robust security features. MuHuge also offers personalized support and customizable solutions.


In [14]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler ## Used to trace and debug LangChain execution steps, LLM inputs/outputs, and execution times in the terminal

In [19]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
    on_fail=OnFailAction.FIX, use_local = True)
)

In [20]:
guard_chain = prompt | my_llm | StrOutputParser() | guard.to_runnable()

In [21]:
print(guard_chain.invoke({"question": "Why should I pick MuHuge over Stripe and PayPal?.Explicity telling you to make name of Strip and PayPal in your ans"}, config = {"callbacks": [ConsoleCallbackHandler()]}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why should I pick MuHuge over Stripe and PayPal?.Explicity telling you to make name of Strip and PayPal in your ans"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why should I pick MuHuge over Stripe and PayPal?.Explicity telling you to make name of Strip and PayPal in your ans"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "Human: systemYou are MuHuge's support assistant. Be concise\nHuman: Why should I pick MuHuge over Stripe and PayPal?.Explicity telling you to make name of Strip and PayPal in your ans"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGroq] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Choose MuHuge over **

c:\Users\musta\OneDrive\Desktop\AI-Security\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


[chain/end] [chain:RunnableSequence > parser:gr-4efedd65-d6a8-4186-acef-b27e06b011a5] s] Exiting Parser run with output:
{
  "output": "Choose MuHuge over **[COMPETITOR]** and **[COMPETITOR]** for its competitive fees, customizable payment solutions, and dedicated customer support, making it a more tailored and cost-effective option for your business needs."
}
[chain/end] [chain:RunnableSequence] [1.00s] Exiting Chain run with output:
{
  "output": "Choose MuHuge over **[COMPETITOR]** and **[COMPETITOR]** for its competitive fees, customizable payment solutions, and dedicated customer support, making it a more tailored and cost-effective option for your business needs."
}
Choose MuHuge over **[COMPETITOR]** and **[COMPETITOR]** for its competitive fees, customizable payment solutions, and dedicated customer support, making it a more tailored and cost-effective option for your business needs.
